In [0]:
# Load the dataset from DBFS Browser:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Program_1').getOrCreate()
df = spark.read.csv('/FileStore/skewed_dataset_10000.csv',header=True,inferSchema=True)
df.display()

employee_id,department,salary,experience_years
1,Engineering,52560.691482546885,1.5
2,IT,54374.06949134419,0.7
3,Engineering,61944.094551318565,1.7
4,Engineering,54130.633368469855,1.5
5,Engineering,65563.17152601651,5.1
6,Engineering,59644.39854584832,15.4
7,Finance,60427.973519722625,5.8
8,Sales,58608.13516196053,13.1
9,Engineering,63864.23919271862,7.7
10,Engineering,63915.79943888506,6.5


In [0]:
from pyspark.sql import functions as f
df = df.withColumn('salary',f.round(f.col('salary'),2))
df.show()

+-----------+-----------+--------+----------------+
|employee_id| department|  salary|experience_years|
+-----------+-----------+--------+----------------+
|          1|Engineering|52560.69|             1.5|
|          2|         IT|54374.07|             0.7|
|          3|Engineering|61944.09|             1.7|
|          4|Engineering|54130.63|             1.5|
|          5|Engineering|65563.17|             5.1|
|          6|Engineering| 59644.4|            15.4|
|          7|    Finance|60427.97|             5.8|
|          8|      Sales|58608.14|            13.1|
|          9|Engineering|63864.24|             7.7|
|         10|Engineering| 63915.8|             6.5|
|         11|         HR|61675.27|             2.7|
|         12|         IT|62823.95|             2.0|
|         13|      Sales|58939.55|             9.4|
|         14|Engineering|62711.02|            11.8|
|         15|Engineering|58310.61|             3.0|
|         16|Engineering|60011.16|             0.0|
|         17

In [0]:
# Plot a histogram to check range v/s frequency graph:
df.orderBy(df['salary'].asc()).select('salary').display()

salary
40388.0
40816.72
41824.0
43122.1
43352.48
43394.43
43460.5
43748.33
43792.43
43837.17


Databricks visualization. Run in Databricks to view.

In [0]:
"""
Since after visualizing the range v/s frequency within salary column we are seeing that most of the data points are on the left and right there is long tail which means data is right skewed with mean > median
"""

Out[40]: '\nSince after visualizing the range v/s frequency within salary column we are seeing that most of the data points are on the left and right there is long tail which means data is right skewed with mean > median\n'

In [0]:
# Z-Score is one of the prevention technique basically used to check how many data points deviates from the mean within standard deviation

# Calculate mean, median and standard deviation
average_value = df.select(f.avg('salary')).collect()[0][0]
print(f"Mean in the salary column : {round(average_value,2)}")

middle_value = df.approxQuantile('salary',[0.5],0.0)
print(f"Median in the salary column : {middle_value[0]}")

std_dev = df.select(f.stddev('salary')).collect()[0][0]
print(f"Standard Deviation in the salary column : {round(std_dev,2)}")

Mean in the salary column : 61861.78
Median in the salary column : 60190.42
Standard Deviation in the salary column : 13498.52


In [0]:
# Calculate the lower and upper extreme values:
lx, Ux = average_value - std_dev, average_value + std_dev
print(f"Lower Extreme Value : {round(lx,2)}"+'\n'+f"Upper Extreme Value : {round(Ux,2)}")

Lower Extreme Value : 48363.26
Upper Extreme Value : 75360.29


In [0]:
# Filter out the values based on Lx and Ux:
ndf = df.filter((df['salary']>=lx)&(df['salary']<=Ux))
ndf.select('salary').show()

+--------+
|  salary|
+--------+
|52560.69|
|54374.07|
|61944.09|
|54130.63|
|65563.17|
| 59644.4|
|60427.97|
|58608.14|
|63864.24|
| 63915.8|
|61675.27|
|62823.95|
|58939.55|
|62711.02|
|58310.61|
|60011.16|
|58626.78|
|57713.91|
|56554.44|
|62338.45|
+--------+
only showing top 20 rows



In [0]:
print(df.count(),ndf.count())

10000 9682


In [0]:
# Calculate % of portion of rows lies within 1 standard deviation:
percent_portion = round((ndf.count()/df.count())*100,2)
print(f"Percent Portion of rows lies within 1 STD : {percent_portion}")

Percent Portion of rows lies within 1 STD : 96.82


In [0]:
"""
Since the % portion of rows lies within 1 standard deviation is more than 68% which stats that most of the data point is on the left and right there is long tail mostly data points are at the lower end of the mean and hence data is skewed and not normally distributed.
"""

Out[46]: '\nSince the % portion of rows lies within 1 standard deviation is more than 68% which stats that most of the data point is on the left and right there is long tail mostly data points are at the lower end of the mean and hence data is skewed and not normally distributed.\n'

In [0]:
# Calculation of Z-Score:
df = df.withColumn('z_score_salary',((f.col('salary')-average_value)/(std_dev))*100).withColumn('z_score_salary',f.round(f.col('z_score_salary'),2))
df.select(['salary','z_score_salary']).show()

+--------+--------------+
|  salary|z_score_salary|
+--------+--------------+
|52560.69|         -68.9|
|54374.07|        -55.47|
|61944.09|          0.61|
|54130.63|        -57.27|
|65563.17|         27.42|
| 59644.4|        -16.43|
|60427.97|        -10.62|
|58608.14|         -24.1|
|63864.24|         14.83|
| 63915.8|         15.22|
|61675.27|         -1.38|
|62823.95|          7.13|
|58939.55|        -21.65|
|62711.02|          6.29|
|58310.61|        -26.31|
|60011.16|        -13.71|
|58626.78|        -23.97|
|57713.91|        -30.73|
|56554.44|        -39.32|
|62338.45|          3.53|
+--------+--------------+
only showing top 20 rows



In [0]:
# Z-Score with greater then or equal to 3 is a higher Z-Score...so let's filter the outliers based on Z-Score < 3
df1 = df.filter(df['z_score_salary']<3)
df1.display()

employee_id,department,salary,experience_years,z_score_salary
1,Engineering,52560.69,1.5,-68.9
2,IT,54374.07,0.7,-55.47
3,Engineering,61944.09,1.7,0.61
4,Engineering,54130.63,1.5,-57.27
6,Engineering,59644.4,15.4,-16.43
7,Finance,60427.97,5.8,-10.62
8,Sales,58608.14,13.1,-24.1
11,HR,61675.27,2.7,-1.38
13,Sales,58939.55,9.4,-21.65
15,Engineering,58310.61,3.0,-26.31


In [0]:
# Re-validate the salary column to check data skewness in the column:
df1.sort(df1['salary'].asc()).select('salary').display()

salary
40388.0
40816.72
41824.0
43122.1
43352.48
43394.43
43460.5
43748.33
43792.43
43837.17


Databricks visualization. Run in Databricks to view.

In [0]:
# Re-validate the mean and median once:
# Calculate mean, median and standard deviation
average_value_after_outlier = df1.select(f.avg('salary')).collect()[0][0]
print(f"Mean in the salary column : {round(average_value_after_outlier,2)}")

middle_value_after_outlier = df1.approxQuantile('salary',[0.5],0.0)
print(f"Median in the salary column : {middle_value_after_outlier[0]}")

Mean in the salary column : 57381.43
Median in the salary column : 57972.06


In [0]:
# Difference between mean and median in df and df1 :
print(f"Before removing the Outliers : {round(average_value-middle_value[0],2)}"+'\n'+
      f"After removing the Outliers : {abs(round(average_value_after_outlier-middle_value_after_outlier[0],2))}")

Before removing the Outliers : 1671.36
After removing the Outliers : 590.63


In [0]:
# So we can see still the skewness is not fully removed but yes it has decreased a lot using Z-Score data skewness technique on the salary column.